
# Four-Format Data Recovery and Validation

## Project Overview

This project demonstrates the recovery, validation, comparison, processing, and export of a historical Gapminder dataset stored in four different formats:

- CSV
- Excel
- Nested JSON
- Parquet

The objective is to recover a consistent canonical dataset from all four sources while identifying invalid records, removing exact duplicates, detecting conflicting records, and verifying exported results.

## Project Objectives

1. Recover data from four file formats.
2. Validate records against the canonical schema.
3. Quarantine invalid records and log exact duplicates.
4. Compare recovered datasets to verify consistency.
5. Perform data analysis using pandas and NumPy.
6. Export results into multiple formats.
7. Verify exported files through round-trip validation.



## 1. Import Libraries and Project Modules

The project uses pandas and the existing modules in the `src` directory. These modules contain the actual implementation of the recovery and validation processes.


In [12]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

from src.readers import read_input
from src.validator import validate_dataset
from src.comparator import compare_datasets
from src.processor import (
    create_canonical_dataset,
    create_asia_analysis,
    create_country_history,
    create_population_index,
    calculate_asian_population,
)
from src.export_validator import verify_all_exports

PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / "config.json", "r", encoding="utf-8") as file:
    config = json.load(file)

with open(PROJECT_ROOT / config["manifest_path"], "r", encoding="utf-8") as file:
    manifest = json.load(file)

REPORTING_YEAR = config.get("reporting_year", 2007)

print("Project modules imported successfully.")
print("Reporting year:", REPORTING_YEAR)
print("Project directory:", PROJECT_ROOT)


Project modules imported successfully.
Reporting year: 2007
Project directory: c:\Users\arnel\Downloads\Four-Format-Data-Recovery-and-Validation



## 2. Data Recovery from Four Formats

The following section reads the four input files using the project's configured readers. Each file is recovered into a pandas DataFrame using the canonical column structure.


In [13]:

recovered_datasets = {}

for settings in manifest["files"]:
    file_path = PROJECT_ROOT / config["data_dir"] / settings["file"]

    recovered = read_input(file_path, settings)

    recovered_datasets[settings["file"]] = recovered

    print(f"\nFile: {settings['file']}")
    print("Rows:", len(recovered))
    print("Columns:", list(recovered.columns))

print("\nAll four formats successfully recovered.")



File: country_records.csv
Rows: 1712
Columns: ['source_file', 'source_record', 'country', 'continent', 'year', 'lifeExp', 'pop', 'gdpPercap']

File: country_records.xlsx
Rows: 1712
Columns: ['source_file', 'source_record', 'country', 'continent', 'year', 'lifeExp', 'pop', 'gdpPercap']

File: country_records.json
Rows: 1712
Columns: ['source_file', 'source_record', 'country', 'continent', 'year', 'lifeExp', 'pop', 'gdpPercap']

File: country_records.parquet
Rows: 1712
Columns: ['source_file', 'source_record', 'country', 'continent', 'year', 'lifeExp', 'pop', 'gdpPercap']

All four formats successfully recovered.



## 3. Data Validation and Cleaning

Each recovered dataset is validated against the canonical schema and manifest rules.

Invalid records are quarantined, exact duplicates are removed and logged, and conflicting country-year keys are rejected.


In [14]:

validation_results = {}
cleaned_datasets = {}

for filename, recovered in recovered_datasets.items():
    result = validate_dataset(recovered, manifest)

    validation_results[filename] = result
    cleaned_datasets[filename] = result.valid_records[
        manifest["canonical_order"]
    ].copy()

    print(f"\nFile: {filename}")
    print("Valid records:", len(result.valid_records))
    print("Quarantined records:", len(result.quarantine))
    print("Duplicates removed:", len(result.duplicates))

print("\nValidation completed for all four datasets.")



File: country_records.csv
Valid records: 1704
Quarantined records: 6
Duplicates removed: 2

File: country_records.xlsx
Valid records: 1704
Quarantined records: 6
Duplicates removed: 2

File: country_records.json
Valid records: 1704
Quarantined records: 6
Duplicates removed: 2

File: country_records.parquet
Valid records: 1704
Quarantined records: 6
Duplicates removed: 2

Validation completed for all four datasets.



## 4. Cross-Format Dataset Comparison

The four recovered datasets are compared to determine whether they contain identical records.

The comparison checks:
- Row counts
- Country and year keys
- Text and integer values
- Floating-point values using an absolute tolerance of 1e-8

The canonical dataset will only be created after all comparisons pass.


In [15]:

comparison_report = compare_datasets(cleaned_datasets)

display(comparison_report)

if not comparison_report["passed"].all():
    raise ValueError("Cross-format comparison failed.")

print("All four datasets match successfully.")


,comparison,expected,actual,passed
0,country_records.csv: row count,1704,1704,True
1,country_records.csv: country-year keys,Same keys as reference,Match,True
2,country_records.csv: country,Exact match,Match,True
3,country_records.csv: continent,Exact match,Match,True
4,country_records.csv: year,Exact match,Match,True
5,country_records.csv: pop,Exact match,Match,True
6,country_records.csv: lifeExp,Difference <= 1e-08,Maximum difference = 0.0,True
7,country_records.csv: gdpPercap,Difference <= 1e-08,Maximum difference = 0.0,True
8,country_records.xlsx: row count,1704,1704,True
9,country_records.xlsx: country-year keys,Same keys as reference,Match,True


All four datasets match successfully.



## 5. Canonical Dataset Creation

After successful comparison, one shared canonical dataset is created from the recovered records.

The dataset is sorted by country and year to provide a consistent structure for subsequent analysis.


In [16]:

canonical = create_canonical_dataset(
    cleaned_datasets["country_records.csv"]
)

print("Canonical dataset shape:", canonical.shape)
print("\nFirst five records:")
display(canonical.head())

print("\nLast five records:")
display(canonical.tail())


Canonical dataset shape: (1704, 6)

First five records:


,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106



Last five records:


,country,continent,year,lifeExp,pop,gdpPercap
1699,Zimbabwe,Africa,1987,62.351,9216418,706.157306
1700,Zimbabwe,Africa,1992,60.377,10704340,693.420786
1701,Zimbabwe,Africa,1997,46.809,11404948,792.449960
1702,Zimbabwe,Africa,2002,39.989,11926563,672.038623
1703,Zimbabwe,Africa,2007,43.487,12311143,469.709298



## 6. Asian Countries Analysis

This analysis selects Asian countries for the configured reporting year.

It calculates population in millions and ranks countries according to life expectancy. The minimum ranking method assigns the same rank to countries with tied life expectancy values.


In [17]:

asia_analysis = create_asia_analysis(
    canonical,
    REPORTING_YEAR
)

print("Reporting year:", REPORTING_YEAR)
print("Number of Asian country records:", len(asia_analysis))

display(asia_analysis.head(10))


Reporting year: 2007
Number of Asian country records: 33


,country,continent,year,lifeExp,pop,gdpPercap,pop_millions,lifeExp_rank
0,Japan,Asia,2007,82.603,127467972,31656.068060,127.467972,1.0
1,"Hong Kong, China",Asia,2007,82.208,6980412,39724.978670,6.980412,2.0
2,Israel,Asia,2007,80.745,6426679,25523.277100,6.426679,3.0
3,Singapore,Asia,2007,79.972,4553009,47143.179640,4.553009,4.0
4,"Korea, Rep.",Asia,2007,78.623,49044790,23348.139730,49.044790,5.0
5,Taiwan,Asia,2007,78.400,23174294,28718.276840,23.174294,6.0
6,Kuwait,Asia,2007,77.588,2505559,47306.989780,2.505559,7.0
7,Oman,Asia,2007,75.640,3204897,22316.192870,3.204897,8.0
8,Bahrain,Asia,2007,75.635,708573,29796.048340,0.708573,9.0
9,Vietnam,Asia,2007,74.249,85262356,2441.576404,85.262356,10.0



## 7. Philippines Historical Analysis

This section examines the Philippines across all available years.

It calculates population in millions and the change in life expectancy relative to the first recorded year (1952).


In [18]:

philippines_history = create_country_history(canonical)

print("Philippines observations:", len(philippines_history))

display(philippines_history)


Philippines observations: 12


,country,continent,year,lifeExp,pop,gdpPercap,pop_millions,lifeExp_change_from_first
0,Philippines,Asia,1952,47.752,22438691,1272.880995,22.438691,0.000
1,Philippines,Asia,1957,51.334,26072194,1547.944844,26.072194,3.582
2,Philippines,Asia,1962,54.757,30325264,1649.552153,30.325264,7.005
3,Philippines,Asia,1967,56.393,35356600,1814.127430,35.356600,8.641
4,Philippines,Asia,1972,58.065,40850141,1989.374070,40.850141,10.313
5,Philippines,Asia,1977,60.060,46850962,2373.204287,46.850962,12.308
6,Philippines,Asia,1982,62.082,53456774,2603.273765,53.456774,14.330
7,Philippines,Asia,1987,64.151,60017788,2189.634995,60.017788,16.399
8,Philippines,Asia,1992,66.458,67185766,2279.324017,67.185766,18.706
9,Philippines,Asia,1997,68.564,75012988,2536.534925,75.012988,20.812



## 8. Population Index Comparison

This section compares the population trends of the Philippines, Indonesia, and Thailand.

The year 1982 serves as the baseline (index = 100). NumPy broadcasting is used to calculate the population index for each country and year.

An independent loop calculation verifies the broadcasting results.


In [19]:

population_index = create_population_index(canonical)

print("Population index table:")
display(population_index)


Population array shape before broadcasting: (3, 6)
Baseline array shape: (3, 1)
Population array shape after broadcasting: (3, 6)
Independent loop verification: PASSED
Population index table:


,country,1982,1987,1992,1997,2002,2007
0,Philippines,100.0,112.273494,125.682418,140.324570,155.256447,170.375577
1,Indonesia,100.0,110.390432,120.524576,129.955720,137.639149,145.782331
2,Thailand,100.0,108.362522,116.056504,123.326192,128.630762,133.262203



## 9. Independent Asian Population Calculation

NumPy Boolean selection is used to identify Asian countries in the reporting year and calculate their combined population.

This calculation provides an independent analytical check against the expected control value in the project manifest.


In [20]:

asian_population = calculate_asian_population(
    canonical,
    REPORTING_YEAR
)

print(
    f"Total Asian population in {REPORTING_YEAR}: "
    f"{asian_population:,}"
)

expected_population = manifest["controls"]["asia_2007_population"]

if REPORTING_YEAR == 2007:
    assert asian_population == expected_population
    print("Expected population control: PASSED")


Total Asian population in 2007: 3,811,953,827
Expected population control: PASSED



## 10. Export Verification

The canonical dataset is exported into CSV, Excel, JSON, and Parquet formats.

The exported files are reloaded and compared against the original canonical dataset to ensure that no data was lost or corrupted during export.


In [21]:

output_dir = PROJECT_ROOT / config["output_dir"]

round_trip_report = verify_all_exports(
    canonical,
    output_dir
)

display(round_trip_report)

print("All exported formats passed round-trip verification.")


,file,expected_rows,actual_rows,result
0,canonical.csv,1704,1704,PASSED
1,canonical.xlsx,1704,1704,PASSED
2,canonical.json,1704,1704,PASSED
3,canonical.parquet,1704,1704,PASSED


All exported formats passed round-trip verification.



## 11. Project Results and Conclusion

The Four-Format Data Recovery and Validation project successfully recovered a historical Gapminder dataset from four different file formats: CSV, Excel, nested JSON, and Parquet.

Each source initially contained 1,712 records. After validation, six invalid records were quarantined and two exact duplicates were removed from each source, resulting in 1,704 valid canonical records per format.

Cross-format comparison confirmed that all four recovered datasets contained consistent records. The resulting canonical dataset was then used for Asian population analysis, Philippines historical analysis, and population indexing for the Philippines, Indonesia, and Thailand.

NumPy broadcasting was applied to calculate population indices, with an independent loop calculation used to verify the results. Finally, the canonical dataset was exported into four formats and subjected to round-trip verification.

Overall, the project demonstrates a reproducible data recovery and validation workflow that preserves data integrity, documents rejected records, detects conflicts, and produces consistent analytical outputs.
